# Does AdaBoost get worse when we add more trees

[Master Tree Ensemble Models course](https://www.trainindata.com/p/master-tree-ensemble-models)

While preparing the boosting section of the course, we found that an AdaBoost that predicts the rent of an apartment becomes worse as we add trees to the ensemble, and this is the opposite of what we expect from an ensemble model.

In this notebook we find out whether this also happens in classification, we repeat the experiment in several datasets, and then we look for the reason.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing, fetch_openml, make_classification
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor
from sklearn.metrics import roc_auc_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

warnings.filterwarnings("ignore")

### The experiment

AdaBoost adds the trees one after the other, so with `staged_predict` we can evaluate the ensemble after each new tree, and we obtain the whole curve by training the model only once.

We write two functions, one for classification and one for regression, that return the performance in the test set after every tree.

In [ ]:
def prepare(X, y):
    X = pd.DataFrame(X).copy()
    for var in X.columns:
        if str(X[var].dtype) in ("object", "category", "str"):
            X[var] = X[var].astype("category").cat.codes
    # AdaBoost does not take missing values, so we replace them by a value outside the distribution
    X = X.astype(float).fillna(-1)
    return X, pd.Series(np.asarray(y))

In [ ]:
def classification_curve(X, y, n_estimators=300, max_depth=1):
    X, y = prepare(X, y)
    y = y.astype("category").cat.codes
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=0
    )
    model = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=max_depth),
        n_estimators=n_estimators,
        random_state=0,
    )
    model.fit(X_train, y_train)
    return pd.Series(
        [roc_auc_score(y_test, p[:, 1]) for p in model.staged_predict_proba(X_test)],
        index=range(1, len(model.estimators_) + 1),
    )


def regression_curve(X, y, n_estimators=100, max_depth=3):
    X, y = prepare(X, y)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
    model = AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=max_depth),
        n_estimators=n_estimators,
        random_state=0,
    )
    model.fit(X_train, y_train)
    return pd.Series(
        [root_mean_squared_error(y_test, p) for p in model.staged_predict(X_test)],
        index=range(1, len(model.estimators_) + 1),
    )

In [ ]:
def summarise(curves, higher_is_better):
    summary = {}
    for name, curve in curves.items():
        best = curve.max() if higher_is_better else curve.min()
        final = curve.iloc[-1]
        summary[name] = {
            "best": best,
            "best at tree": curve.idxmax() if higher_is_better else curve.idxmin(),
            "final": final,
            "lost from best %": 100 * abs(best - final) / best,
        }
    return pd.DataFrame(summary).T.round(4)

### Classification

We start with data that we make ourselves, so that we control how hard the problem is. We train one model with clean labels, and then we flip the labels of ten and twenty percent of the observations, because the weakness that everybody mentions about AdaBoost is that it chases the observations that it predicts badly, and a wrong label is an observation that no model can ever predict well.

In [ ]:
X, y = make_classification(n_samples=6000, n_features=20, n_informative=6, random_state=0)

curves = {"clean labels": classification_curve(X, y)}
for noise in [0.1, 0.2]:
    rng = np.random.default_rng(0)
    noisy = np.where(rng.random(len(y)) < noise, 1 - y, y)
    curves[f"{int(noise * 100)}% wrong labels"] = classification_curve(X, noisy)

synthetic_curves = pd.DataFrame(curves)
summarise(curves, higher_is_better=True)

In [ ]:
synthetic_curves.plot()
plt.xlabel("Number of trees")
plt.ylabel("roc-auc in the test set")
plt.title("AdaBoost with data that we made ourselves")
plt.show()

The labels with noise bring the curves down, which is expected, and yet none of the curves turns downwards as we add trees. Let us now try with real data.

In [ ]:
datasets = {
    "credit-g": 31,
    "diabetes": 37,
    "spambase": 44,
    "phoneme": 1489,
    "churn": 40701,
}

curves = {}
for name, data_id in datasets.items():
    data = fetch_openml(data_id=data_id, as_frame=True)
    curves[name] = classification_curve(data.data, data.target)

classification_curves = pd.DataFrame(curves)
summarise(curves, higher_is_better=True)

In [ ]:
classification_curves.plot()
plt.xlabel("Number of trees")
plt.ylabel("roc-auc in the test set")
plt.title("AdaBoost in five classification datasets")
plt.legend(loc="lower right")
plt.show()

The performance improves quickly with the first trees and then it flattens, and the models that lose something from their best point lose less than two percent of it. In classification we can therefore add trees without fear.

### Regression

Now we repeat the experiment with the rent of the apartments, and with the house prices of California, which is a dataset that comes with Scikit-learn.

In [ ]:
rent = pd.read_csv("../prepare-data/apartment_rent.csv", index_col="time")
california = fetch_california_housing(as_frame=True)

curves = {
    "apartment rent": regression_curve(rent.drop(columns="price"), rent["price"]),
    "california housing": regression_curve(california.data, california.target),
}

summarise(curves, higher_is_better=False)

In [ ]:
# the two targets are in different units, so we divide each curve by its own best value
relative = pd.DataFrame({name: curve / curve.min() for name, curve in curves.items()})

relative.plot()
plt.xlabel("Number of trees")
plt.ylabel("rmse divided by the best rmse")
plt.title("AdaBoost in two regression datasets")
plt.show()

Both curves reach their best point with around ten trees, and from there they climb. The rent model ends up more than fifty percent worse than its best point, and the California model more than fifteen percent, so this is not something particular to our apartments.

### Why the two behave so differently

In classification a mistake is a yes or a no, so every observation that the model gets wrong receives the same treatment, and no observation can weigh more than another one however wrong the model is about it.

In regression the error is a number, and AdaBoost divides the error of every observation by the largest error of that round. One apartment of 52,500 against a median of 1,350 makes the error of everybody else look like zero, so the weights concentrate on a handful of extreme apartments, and the model follows them.

If this is the reason, then the curve should behave when we remove the extreme rents, and also when we take the logarithm of the rent, which brings the expensive apartments closer to the rest.

In [ ]:
print(f"skew of the rent: {rent['price'].skew():.1f}")
print(f"the most expensive apartment costs {rent['price'].max() / rent['price'].median():.0f} times the median")

In [ ]:
X = rent.drop(columns="price")
y = rent["price"]
affordable = y < y.quantile(0.99)

curves = {
    "all the apartments": regression_curve(X, y),
    "without the top 1%": regression_curve(X[affordable], y[affordable]),
    "logarithm of the rent": regression_curve(X, np.log(y)),
}

summarise(curves, higher_is_better=False)

In [ ]:
relative = pd.DataFrame({name: curve / curve.min() for name, curve in curves.items()})

relative.plot()
plt.xlabel("Number of trees")
plt.ylabel("rmse divided by the best rmse")
plt.title("The same data with the extreme rents under control")
plt.show()

Removing the one percent of the most expensive apartments halves the damage, and taking the logarithm of the rent almost removes it, so the long tail of the target is indeed what breaks the model.

### What we take home

In classification we can add trees to an AdaBoost and the performance flattens, so the number of trees is a parameter that we can set generously.

In regression the number of trees is a real hyperparameter that we need to tune, because AdaBoost measures the error of each observation relative to the largest error of the round, and a target with a long tail sends the weights to the extreme observations.

When the target has a long tail, the logarithm of the target, or a loss that grows more slowly, are cheaper solutions than training more trees.